# The pyBME MCP Server: Agent-Driven Uncertainty Reasoning

> **What this notebook shows**: how an LLM agent uses the pyBME MCP
> server to orchestrate a complete BME workflow — from data ingest
> through operator comparison to optimal sensor placement —
> **without the user writing any pyBME code**.

## Why wrap pyBME in an MCP server?

The library notebook
([10_course_demo_bme_to_hodge.ipynb](10_course_demo_bme_to_hodge.ipynb))
shows the full pyBME API — covariance fitting, BME prediction, network
operators.  Every step is ~5–20 lines of Python.

The **MCP server** lifts the interface from *code* to *intent*:

| Without MCP (library) | With MCP (agent) |
|----------------------|------------------|
| `fit_covariance(ch, zh, model="exponential")` | *"Fit an uncertainty model to my PM2.5 data"* |
| Build adjacency, instantiate `NetworkCovariance`, call `bme_predict_network` | *"Run a network-aware update using the graph Laplacian"* |
| Write `for` loops to compare 3 operator families | *"Compare Euclidean vs graph vs physics-informed operators"* |
| Manually compute variance reduction | *"Explain what's driving uncertainty at the outlet"* |

The seven MCP tools form a **reasoning pipeline**:

```
ingest_external_scenario_evidence → inspect_modeling_context
       → fit_uncertainty_model → run_uncertainty_update
       → explain_uncertainty_drivers
       → compare_operator_approaches
       → design_next_observation_or_scenario
```

This notebook simulates the agent's tool calls so you can see
the inputs, outputs, and stateful registry that the server maintains.


In [ ]:
import json
import numpy as np

# Import MCP server components directly (no transport needed for demo)
from pybme_mcp.server import (
    inspect_modeling_context,
    fit_uncertainty_model,
    run_uncertainty_update,
    explain_uncertainty_drivers,
    compare_operator_approaches,
    design_next_observation_or_scenario,
    ingest_external_scenario_evidence,
)
from pybme_mcp.schemas import (
    InspectModelingContextRequest,
    FitUncertaintyModelRequest,
    RunUncertaintyUpdateRequest,
    ExplainUncertaintyDriversRequest,
    CompareOperatorApproachesRequest,
    DesignNextObservationOrScenarioRequest,
    IngestExternalScenarioEvidenceRequest,
    HardObservationSet,
    SoftObservation,
    NetworkSpec,
    EstimationTargets,
    OperatorAlternative,
    CandidateObservation,
)
from pybme_mcp.registry import registry

def show(obj):
    """Pretty-print a Pydantic model as JSON."""
    print(json.dumps(obj.model_dump(), indent=2, default=str))

print("MCP server components loaded. Registry is empty:")
print(f"  models:      {len(registry.models)}")
print(f"  scenarios:   {len(registry.scenarios)}")
print(f"  evidence:    {len(registry.evidence)}")
print(f"  comparisons: {len(registry.comparisons)}")


---
## Scenario: Pollutant Monitoring in a River Network

An agent receives this user message:

> *"I have concentration readings at 3 sensors in a small river network.
> Tell me what's happening at the unmetered junctions and the outlet."*

The agent doesn't know pyBME's API.  It has access to 7 MCP tools.
Let's follow its reasoning chain.


### Step 1: `ingest_external_scenario_evidence`

The agent's first move is to bundle the data—hard observations,
network topology—into a reusable evidence bundle.

> **Agent thinks**: *"The user gave me node-indexed data on a directed network.
> I'll ingest everything so later tools can reference it by ID."*


In [ ]:
# ── Agent calls: ingest_external_scenario_evidence ───────────
ingest_response = ingest_external_scenario_evidence(
    IngestExternalScenarioEvidenceRequest(
        hard_data=HardObservationSet(
            node_indices=[0, 1, 5],
            values=[8.5, 3.2, 5.0],
            labels=["Headwater-A", "Headwater-B", "Tributary"],
        ),
        network=NetworkSpec(
            n_nodes=8,
            directed_edges=[[0,2],[1,2],[2,3],[3,4],[5,4],[4,6],[6,7]],
            description="Synthetic river: 2 headwaters merge, tributary joins mid-reach, outlet at node 7",
        ),
        source="field_sensors",
        scenario_label="baseline_dry_weather",
    )
)

print("=== ingest response ===")
show(ingest_response)
evidence_id = ingest_response.evidence_id
print(f"\n→ Evidence bundle stored as: {evidence_id}")
print(f"→ Recommended next tools: {ingest_response.recommended_next_tools}")


The server stored the data and returned an `evidence_id`.
Subsequent tools reference this ID instead of re-sending raw arrays.
This is how the agent maintains conversational state.

### Step 2: `inspect_modeling_context`

> **Agent thinks**: *"Before fitting anything, let me classify the
> problem and see what model families make sense."*


In [ ]:
# ── Agent calls: inspect_modeling_context ─────────────────────
context_response = inspect_modeling_context(
    InspectModelingContextRequest(
        hard_data=HardObservationSet(
            node_indices=[0, 1, 5],
            values=[8.5, 3.2, 5.0],
        ),
        network=NetworkSpec(
            n_nodes=8,
            directed_edges=[[0,2],[1,2],[2,3],[3,4],[5,4],[4,6],[6,7]],
        ),
        domain_hint="river water-quality monitoring",
    )
)

print("=== modeling context ===")
show(context_response)
print(f"\n→ Problem type: {context_response.problem_type}")
print(f"→ Candidate priors: {context_response.candidate_priors}")
print(f"→ Risks: {context_response.risks}")


The server classified this as a **network_spatial** problem and
recommended `graph_laplacian` and `physics_informed_network` as
candidate priors.  It also flagged that operator assumptions should
be compared explicitly.

### Step 3: `fit_uncertainty_model`

> **Agent thinks**: *"I'll start with the graph Laplacian — the simplest
> network prior.  I can reference the evidence bundle by ID."*


In [ ]:
# ── Agent calls: fit_uncertainty_model ────────────────────────
fit_response = fit_uncertainty_model(
    FitUncertaintyModelRequest(
        name="river_laplacian",
        model_family="graph_laplacian",
        evidence_id=evidence_id,
        covariance_model="exponential",
        operator_hyperparameters={"kappa": 1.0, "sigma2": 4.0},
    )
)

print("=== fit response ===")
show(fit_response)
model_id = fit_response.model_id
print(f"\n→ Model registered as: {model_id}")
print(f"→ Family: {fit_response.model_family}")
print(f"→ Implementation status: {fit_response.implementation_status}")
print(f"→ Cross-validation RMSE: {fit_response.fit_quality.get('rmse', 'N/A')}")


The model is now **registered in the server's stateful registry**.
Any subsequent tool can reference it by `model_id`.

### Step 4: `run_uncertainty_update`

> **Agent thinks**: *"Now predict at the 5 unmetered nodes."*


In [ ]:
# ── Agent calls: run_uncertainty_update ───────────────────────
update_response = run_uncertainty_update(
    RunUncertaintyUpdateRequest(
        model_id=model_id,
        estimation_targets=EstimationTargets(
            node_indices=[2, 3, 4, 6, 7],
            labels=["Junction", "Mid-reach", "Junction-2", "Lower", "Outlet"],
        ),
    )
)

print("=== update response ===")
scenario_id = update_response.scenario_id
print(f"Scenario: {scenario_id}")
print(f"Implementation: {update_response.implementation_status}")
if update_response.posterior:
    labels = ["Junction", "Mid-reach", "Junction-2", "Lower", "Outlet"]
    print(f"\n{'Target':>12s}  {'Mean':>8s}  {'Std':>8s}  {'95% CI':>18s}")
    print("-" * 52)
    for i, lbl in enumerate(labels):
        m = update_response.posterior.mean[i]
        v = update_response.posterior.variance[i]
        lo = update_response.posterior.ci_lower[i]
        hi = update_response.posterior.ci_upper[i]
        print(f"{lbl:>12s}  {m:8.2f}  {np.sqrt(v):8.2f}  [{lo:7.2f}, {hi:7.2f}]")


The server computed posteriors at all 5 unmetered nodes and tracked
the scenario.  Notice the agent never wrote `bme_predict_network`,
`NetworkCovariance`, or `adjacency_from_edges` — the MCP tool
handled all of that.

### Step 5: `explain_uncertainty_drivers`

> **Agent thinks**: *"The outlet has the widest CI.  Let me explain why."*


In [ ]:
# ── Agent calls: explain_uncertainty_drivers ──────────────────
explain_response = explain_uncertainty_drivers(
    ExplainUncertaintyDriversRequest(
        scenario_id=scenario_id,
    )
)

print("=== explanation ===")
show(explain_response)
print(f"\n→ Dominant drivers:")
for d in explain_response.dominant_drivers:
    print(f"    • {d}")
print(f"\n→ Recommendations:")
for r in explain_response.recommendations:
    print(f"    • {r}")


The server analysed the posterior and explained *why* uncertainty
is distributed the way it is — which nodes are data-rich, which are
topologically distant from sensors, and what would help most.

### Step 6: `design_next_observation_or_scenario`

> **Agent thinks**: *"If the user can add one more sensor,
> where should it go?"*


In [ ]:
# ── Agent calls: design_next_observation_or_scenario ──────────
design_response = design_next_observation_or_scenario(
    DesignNextObservationOrScenarioRequest(
        model_id=model_id,
        candidates=[
            CandidateObservation(node_index=2, label="Junction"),
            CandidateObservation(node_index=3, label="Mid-reach"),
            CandidateObservation(node_index=4, label="Junction-2"),
            CandidateObservation(node_index=6, label="Lower"),
            CandidateObservation(node_index=7, label="Outlet"),
        ],
    )
)

print("=== sensor placement ranking ===")
for i, entry in enumerate(design_response.ranking, 1):
    print(f"  #{i}  node {entry['node_index']}  "
          f"({entry.get('label', '?'):>12s})  "
          f"score = {entry.get('score', 'N/A'):.3f}  "
          f"— {entry.get('reason', '')[:60]}")
print(f"\nRationale:")
for r in design_response.rationale:
    print(f"  • {r}")


### Step 7: `compare_operator_approaches`

> **Agent thinks**: *"The context tool suggested comparing operators.
> Let me pit graph Laplacian against physics-informed against Euclidean."*


In [ ]:
# ── Agent calls: compare_operator_approaches ──────────────────
compare_response = compare_operator_approaches(
    CompareOperatorApproachesRequest(
        hard_data=HardObservationSet(
            node_indices=[0, 1, 5],
            values=[8.5, 3.2, 5.0],
        ),
        network=NetworkSpec(
            n_nodes=8,
            directed_edges=[[0,2],[1,2],[2,3],[3,4],[5,4],[4,6],[6,7]],
        ),
        alternatives=[
            OperatorAlternative(
                label="Graph Laplacian",
                model_family="graph_laplacian",
                operator_hyperparameters={"kappa": 1.0, "sigma2": 4.0},
            ),
            OperatorAlternative(
                label="Physics-Informed",
                model_family="physics_informed_network",
                operator_hyperparameters={"kappa": 1.0, "sigma2": 4.0, "alpha": 1.0, "lam": 2.0},
            ),
        ],
        estimation_targets=EstimationTargets(
            node_indices=[2, 3, 4, 6, 7],
            labels=["Junction", "Mid-reach", "Junction-2", "Lower", "Outlet"],
        ),
    )
)

print("=== operator comparison ===")
print(f"Question: {compare_response.question}")
print(f"\nComparison ID: {compare_response.comparison_id}")
for alt in compare_response.alternatives:
    print(f"\n  {alt['label']}:")
    for k, v in alt.items():
        if k != "label":
            print(f"    {k}: {v}")
print(f"\nRecommended: {compare_response.recommended_choice.get('label', 'N/A')}")
print(f"  Reason: {compare_response.recommended_choice.get('reason', 'N/A')}")


---
## The Stateful Registry

After this 7-tool chain, the server's registry contains the full
audit trail.  An agent (or human) can query it at any time via
MCP resources.


In [ ]:
print(f"Registry state after 7 tool calls:")
print(f"  Evidence bundles  : {len(registry.evidence)}")
print(f"  Fitted models     : {len(registry.models)}")
print(f"  Posterior scenarios: {len(registry.scenarios)}")
print(f"  Comparisons       : {len(registry.comparisons)}")
print(f"  Action history    : {len(registry.history)} entries")
print()
print("History log:")
for entry in registry.history:
    print(f"  [{entry['timestamp']}] {entry['action']}: {entry.get('id', '')}")


---
## What the User Actually Sees

The user never sees tool calls or JSON schemas.  They see the agent's
natural-language synthesis:

> **User**: I have concentration readings at 3 sensors in a small river network.
> Tell me what's happening at the unmetered junctions and the outlet.
>
> **Agent**: I analysed your 8-node river network using a graph Laplacian
> prior (κ=1.0, σ²=4.0).  Here's what I found:
>
> | Node | Predicted | 95% CI | Confidence |
> |------|-----------|--------|------------|
> | Junction (2) | 5.8 mg/L | [3.1, 8.5] | Good — directly downstream of both sensors |
> | Mid-reach (3) | 5.4 mg/L | [2.5, 8.3] | Moderate |
> | Junction-2 (4) | 5.3 mg/L | [2.2, 8.4] | Moderate — tributary data helps |
> | Lower (6) | 5.1 mg/L | [1.8, 8.4] | Lower — 2 hops from nearest sensor |
> | Outlet (7) | 5.0 mg/L | [1.4, 8.6] | Lowest — furthest from all sensors |
>
> The outlet has the widest uncertainty because it's topologically
> distant from all three sensors.  If you can add one more sensor,
> **node 6 (Lower)** would reduce overall network uncertainty the most.
>
> I also compared graph Laplacian vs physics-informed operators.
> The physics-informed model produces tighter confidence intervals
> at Junction-2 because it enforces mass-balance consistency with
> upstream data.

Behind the scenes, the agent made 7 MCP tool calls.  The user wrote
zero lines of code.

---
## Summary

| MCP Tool | What it does | Why it matters |
|----------|-------------|----------------|
| `ingest_external_scenario_evidence` | Bundles data + topology into a reusable ID | Conversational state: "use my earlier data" |
| `inspect_modeling_context` | Classifies the problem, recommends families | Agent knows which tools to call next |
| `fit_uncertainty_model` | Fits/registers a model by intent, not API | "Fit a graph Laplacian" → 1 call vs ~10 lines |
| `run_uncertainty_update` | Computes posteriors at target locations | Full BME prediction as a single tool call |
| `explain_uncertainty_drivers` | Narrates what's driving uncertainty | Agent can explain *why*, not just *what* |
| `compare_operator_approaches` | Pit operator families against each other | Automated model selection with rationale |
| `design_next_observation_or_scenario` | Ranks candidate sensors by value | Optimal monitoring design as a service |

The MCP server turns pyBME from a **library you code against**
into a **reasoning service an agent orchestrates**.
